In [1]:
import pandas as pd
import os

checkpoint_path = r"C:\Users\Priyanka\OneDrive\Desktop\ocr_checkpoint.csv"

checkpoint = pd.read_csv(checkpoint_path)

print("Checkpoint rows:", len(checkpoint))
print("Columns:", checkpoint.columns.tolist())
print("\nLast 5 rows:")
display(checkpoint.tail())

Checkpoint rows: 1300
Columns: ['filename', 'category', 'filepath', 'ocr_text']

Last 5 rows:


,filename,category,filepath,ocr_text
1295,IMG-20260813-WA0368.jpg,Receipts_Bills,C:\Users\Priyanka\OneDrive\Desktop\Smart_Scree...,243 M @ NR 123 56+ #ll 42% TAYLOR SSWIF THE ER...
1296,IMG-20260813-WA0371.jpg,Receipts_Bills,C:\Users\Priyanka\OneDrive\Desktop\Smart_Scree...,THE LFE OF SHOIHRL ORDER #0073 FOR TAVLOR SUNI...
1297,IMG-20260813-WA0372.jpg,Receipts_Bills,C:\Users\Priyanka\OneDrive\Desktop\Smart_Scree...,"ORDER #0001 FOR BETSYHART TUESDAY_ JANUARY 31,..."
1298,IMG-20260813-WA0374.jpg,Receipts_Bills,C:\Users\Priyanka\OneDrive\Desktop\Smart_Scree...,242 M @ NR 5858 56+ all 42% FOLKLORE (DELUXE V...
1299,IMG-20260813-WA0375.jpg,Receipts_Bills,C:\Users\Priyanka\OneDrive\Desktop\Smart_Scree...,"Taylor Swift Midnights DATe: OCTOBER 21, 2022 ..."


In [3]:
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))


CUDA: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [4]:
import time
import pandas as pd
import easyocr

# ==============================
# FILE PATHS
# ==============================
metadata_path = r"C:\Users\Priyanka\OneDrive\Desktop\metadata.csv"
checkpoint_path = r"C:\Users\Priyanka\OneDrive\Desktop\ocr_checkpoint.csv"
output_path = r"C:\Users\Priyanka\OneDrive\Desktop\metadata_ocr.csv"

# ==============================
# LOAD ORIGINAL + CHECKPOINT
# ==============================
df = pd.read_csv(metadata_path)
checkpoint = pd.read_csv(checkpoint_path)

completed = len(checkpoint)

print(f"Total screenshots: {len(df)}")
print(f"Already completed: {completed}")
print(f"Remaining: {len(df) - completed}")

# ==============================
# GPU OCR
# ==============================
print("\nLoading EasyOCR GPU model...")
reader = easyocr.Reader(['en'], gpu=True)

print("GPU OCR ready!")
print("Resuming OCR...\n", flush=True)

# Keep previous OCR results
ocr_texts = checkpoint["ocr_text"].tolist()

start_time = time.time()

# ==============================
# RESUME FROM CHECKPOINT
# ==============================
for i in range(completed, len(df)):

    row = df.iloc[i]
    image_path = row["filepath"]

    try:
        result = reader.readtext(image_path, detail=0)
        text = " ".join(result)

    except Exception as e:
        text = ""
        print(f"\nERROR at {i+1}: {image_path}")
        print(e, flush=True)

    ocr_texts.append(text)

    processed = i + 1

    # Progress every 10 images
    if processed % 10 == 0 or processed == completed + 1:

        elapsed = time.time() - start_time
        done_now = processed - completed
        avg_time = elapsed / done_now
        remaining = len(df) - processed
        eta = avg_time * remaining

        print(
            f"Processed {processed}/{len(df)} | "
            f"Remaining: {remaining} | "
            f"ETA: {eta/60:.1f} min",
            flush=True
        )

    # ==============================
    # CHECKPOINT EVERY 100
    # ==============================
    if processed % 100 == 0:

        checkpoint_df = df.iloc[:processed].copy()
        checkpoint_df["ocr_text"] = ocr_texts

        checkpoint_df.to_csv(
            checkpoint_path,
            index=False
        )

        print(
            f"✓ CHECKPOINT SAVED: {processed}/{len(df)}",
            flush=True
        )

# ==============================
# FINAL FILE
# ==============================
final_df = df.copy()
final_df["ocr_text"] = ocr_texts

final_df.to_csv(
    output_path,
    index=False
)

total_time = time.time() - start_time

print("\n====================================")
print("          OCR COMPLETED")
print("====================================")
print("Total screenshots:", len(final_df))
print("Resume time:", round(total_time / 60, 2), "minutes")
print("Final file:", output_path)
print("====================================")

Total screenshots: 2392
Already completed: 1300
Remaining: 1092

Loading EasyOCR GPU model...
GPU OCR ready!
Resuming OCR...

Processed 1301/2392 | Remaining: 1091 | ETA: 67.4 min
Processed 1310/2392 | Remaining: 1082 | ETA: 58.0 min
Processed 1320/2392 | Remaining: 1072 | ETA: 46.4 min
Processed 1330/2392 | Remaining: 1062 | ETA: 38.1 min
Processed 1340/2392 | Remaining: 1052 | ETA: 35.3 min
Processed 1350/2392 | Remaining: 1042 | ETA: 33.5 min
Processed 1360/2392 | Remaining: 1032 | ETA: 33.1 min
Processed 1370/2392 | Remaining: 1022 | ETA: 31.7 min
Processed 1380/2392 | Remaining: 1012 | ETA: 30.9 min
Processed 1390/2392 | Remaining: 1002 | ETA: 29.8 min
Processed 1400/2392 | Remaining: 992 | ETA: 29.3 min
✓ CHECKPOINT SAVED: 1400/2392
Processed 1410/2392 | Remaining: 982 | ETA: 28.6 min
Processed 1420/2392 | Remaining: 972 | ETA: 28.3 min
Processed 1430/2392 | Remaining: 962 | ETA: 27.5 min
Processed 1440/2392 | Remaining: 952 | ETA: 26.8 min
Processed 1450/2392 | Remaining: 942 | 